# STAT163 · Week 2 · Before the lecture: DataFrames and profiling

This notebook takes about **50 minutes**. Work through it before the lecture.

The tools:
how to pick rows and columns, how to sort and make new columns, what a column is stored as, and a
fixed sequence of checks to run on any table before you compute anything. That sequence
is called **profiling**: describing a table before you use it. If you did the basics
notebook, Part 1 will be quick.

**How to use it.** Run the cells one at a time, from top to bottom. Three kinds of cells:

- **Read and run.** Run the cell and read the output. Most cells are this kind.
- **Predict.** Before you run the cell, write what you expect in the comment line.
  Then run it and compare. The answer sits under **Answer** below the cell; open it
  after you run.
- **Try it.** Change one thing in the code and run it again.

Three cells stop with an error on purpose, and one prints a warning in red and carries
on.

Nothing here is submitted or graded.

AI is welcome in this notebook. Ask it to explain a cell, or to explain why your
prediction was wrong.

## Before you start

New to notebooks or to pandas, or a year away from Python? Run the
[basics notebook](https://github.com/stat163-2026t1/week2-pre-lecture/blob/main/00-pandas-basics.ipynb)
in this repository first, about 45 minutes, then come back here.

## Load the table

One month of sales from an online shop, 78 015 rows. One row is one line on one receipt:
one product, bought once. The columns: `Invoice` is the receipt number, `StockCode` the product code,
`Description` the product name, `Quantity` how many units, `InvoiceDate` when,
`Price` per unit, `Customer ID` who bought, `Country` where it was shipped. An invoice number that starts with C marks a
cancellation; the shop's documentation says so.

The table is read straight from the web.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/stat163-2026t1/week2-pre-lecture/main/data/online_retail_2010_11.csv"
df = pd.read_csv(url)
df.shape

(78015, 8)

In [2]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,529995,48184,DOORMAT ENGLISH ROSE,6,2010-11-01 08:56:00,7.95,16316.0,United Kingdom
1,529995,48187,DOORMAT NEW ENGLAND,4,2010-11-01 08:56:00,7.95,16316.0,United Kingdom
2,529995,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2010-11-01 08:56:00,6.75,16316.0,United Kingdom
3,529995,22708,WRAP DOLLY GIRL,25,2010-11-01 08:56:00,0.42,16316.0,United Kingdom
4,529995,22781,GUMBALL MAGAZINE RACK,4,2010-11-01 08:56:00,7.65,16316.0,United Kingdom


## Part 1 — Picking rows and columns

There are three ways to ask a DataFrame for a part of itself, and all three use square
brackets. The first two differ by one character.

In [ ]:
# Predict: what type comes out of each? Write your guess here:
print(type(df["Country"]))
print(type(df[["Country"]]))

<details>
<summary>Answer</summary>

One pair of brackets with a name gives a Series. A list inside the brackets gives a
DataFrame, even when the list holds one name.

</details>

| You want | Write | You get |
|---|---|---|
| one column | `df["Country"]` | a Series |
| several columns | `df[["Invoice", "Country"]]` | a DataFrame |
| the rows where a condition holds | `df[df["Quantity"] < 0]` | a DataFrame |

### A condition gives a boolean mask

On its own, `df["Quantity"] < 0` is a Series of `True` and `False`, one per row, with the
same index as `df`. The comparison you write is the **condition**; the Series it produces
is the **boolean mask**, or mask for short. A mask lies over the table and lets a row
through where it is `True`: put it inside `df[...]` and you keep those rows.

In [4]:
mask = df["Quantity"] < 0
mask.head()

0    False
1    False
2    False
3    False
4    False
Name: Quantity, dtype: bool

In [ ]:
# Predict: what does this number count?
mask.sum()

<details>
<summary>Answer</summary>

`True` counts as 1 and `False` as 0, so `.sum()` on a mask counts the rows where the
condition holds. `.mean()` gives the share: the fraction of rows that match. This is the
usual way to count in pandas: write the condition, then sum its mask. The count prints with its storage
type, `np.int64(1419)`: `np` is NumPy, the library pandas stores its numbers with, and
`print` would show the bare number.

</details>

In [6]:
(df["Quantity"] < 0).mean().round(4)

np.float64(0.0182)

### Combining conditions

`&` means and, `|` means or, `~` means not, `!=` means not equal. Python's own `and` and
`or` do not work here: they need one True or False, and a mask holds one per row. Put
every condition in parentheses. On plain numbers:

In [7]:
print(2 > 1 & 3 > 2)
print((2 > 1) & (3 > 2))

False
True


Python does `&` before `>`, so the first line becomes `2 > 1 > 2` (`1 & 3` is 1), which
Python reads as 2 > 1 and 1 > 2, and that is False. With parentheses the two comparisons
happen first.

In [8]:
df[(df["Country"] == "Germany") & (df["Quantity"] > 100)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
7970,530799,21637,ASSORTED SANSKRIT MINI NOTEBOOK,288,2010-11-04 12:31:00,1.06,12497.0,Germany
7971,530799,20759,CHRYSANTHEMUM POCKET BOOK,240,2010-11-04 12:31:00,0.64,12497.0,Germany
7973,530799,20770,ABSTRACT CIRCLE JOURNAL,108,2010-11-04 12:31:00,2.10,12497.0,Germany
7975,530799,20758,ABSTRACT CIRCLES POCKET BOOK,240,2010-11-04 12:31:00,0.64,12497.0,Germany
7976,530799,20772,GARDEN PATH JOURNAL,108,2010-11-04 12:31:00,2.10,12497.0,Germany
7978,530799,20760,GARDEN PATH POCKET BOOK,480,2010-11-04 12:31:00,0.64,12497.0,Germany
7981,530799,20756,GREEN FERN POCKET BOOK,240,2010-11-04 12:31:00,0.64,12497.0,Germany
7984,530799,20755,BLUE PAISLEY POCKET BOOK,240,2010-11-04 12:31:00,0.64,12497.0,Germany
7987,530799,20978,36 PENCILS TUBE SKULLS,128,2010-11-04 12:31:00,1.06,12497.0,Germany
7989,530799,20979,36 PENCILS TUBE RED RETROSPOT,128,2010-11-04 12:31:00,1.06,12497.0,Germany


`isin` asks whether each value is in the list.

In [9]:
df[df["Country"].isin(["France", "Germany", "Netherlands"])].shape

(2462, 8)

**Try it.** Count the rows shipped outside the United Kingdom that have a negative
quantity.

In [10]:
# Try it: one condition on Country, one on Quantity, & between them, .sum() at the end

### Labels and positions

Every row has a label in the index. `.loc[label]` picks by label. `.iloc[position]` picks
by position, counting from 0. In a freshly loaded table the label and the position are the
same number, so the difference is invisible. It becomes visible after sorting.
`sort_values` sorts by a column; `ascending=False` puts the largest first.

In [11]:
top = df.sort_values("Quantity", ascending=False).head(3)
top

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
7253,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838.0,United Kingdom
7254,530715,17003,BROCADE RING PURSE,6336,2010-11-04 11:36:00,0.19,15838.0,United Kingdom
7250,530714,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4320,2010-11-04 11:35:00,0.18,16754.0,United Kingdom


Look at the index of `top`. The labels travelled with the rows.

In [ ]:
# Predict: which row does each line return — the first row of top, or another one?
print(top.iloc[0]["Description"])
print(top.loc[7253]["Description"])

<details>
<summary>Answer</summary>

The same row twice. `.iloc[0]` means the first row of `top`. `.loc[7253]` means the row
whose label is 7253, and that is the same row, because the label travelled with it.

</details>

In [ ]:
# Predict: what happens here?
top.loc[0]

<details>
<summary>Answer</summary>

A `KeyError`. No row in `top` has the label 0. The label 0 belongs to the first row of
`df`, and that row is not among the top three.

</details>

When you want positions after sorting, use `.iloc`, or reset the index so that labels
and positions agree again. `drop=True` throws the old labels away instead of keeping them
as a column:

In [14]:
top = top.reset_index(drop=True)
top.loc[0]

Invoice                                     530715
StockCode                                    84347
Description    ROTATING SILVER ANGELS T-LIGHT HLDR
Quantity                                      9360
InvoiceDate                    2010-11-04 11:36:00
Price                                         1.69
Customer ID                                15838.0
Country                             United Kingdom
Name: 0, dtype: object

`.loc` also takes a condition and a list of columns at once.

In [15]:
df.loc[df["Country"] == "Germany", ["Invoice", "Description", "Quantity"]].head()

,Invoice,Description,Quantity
1599,530122,POSTAGE,4
1600,530122,RETROSPOT LAMP,3
1601,530122,RED 3 PIECE RETROSPOT CUTLERY SET,6
1602,530122,PACK OF 60 DINOSAUR CAKE CASES,24
1603,530122,PACK OF 72 RETROSPOT CAKE CASES,24


## Part 2 — Sorting and new columns

In [16]:
df.sort_values("Price", ascending=False).head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
15765,C531400,AMAZONFEE,AMAZON FEE,-1,2010-11-08 10:08:00,6706.71,NaN,United Kingdom
15775,531411,AMAZONFEE,AMAZON FEE,1,2010-11-08 10:11:00,6706.71,NaN,United Kingdom
40,C529999,M,Manual,-1,2010-11-01 09:11:00,1435.29,NaN,RSA
39,529998,M,Manual,1,2010-11-01 09:09:00,1435.29,NaN,RSA
25895,C532255,D,Discount,-1,2010-11-11 12:50:00,1269.51,14088.0,United Kingdom


`sort_values` returns a new table; `df` itself is unchanged. Sorting by one column keeps
every row and changes the order. Sorting by several columns sorts by the first, and rows
that are equal on the first are ordered by the second. `ascending` takes one True or
False per column.

In [17]:
df.sort_values(["Country", "Price"], ascending=[True, False]).head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
74220,536009,22509,SEWING BOX RETROSPOT DESIGN,4,2010-11-29 15:07:00,14.95,12415.0,Australia
74243,536009,22846,BREAD BIN DINER STYLE RED,16,2010-11-29 15:07:00,14.95,12415.0,Australia
74244,536009,22848,BREAD BIN DINER STYLE PINK,8,2010-11-29 15:07:00,14.95,12415.0,Australia
74245,536009,22847,BREAD BIN DINER STYLE IVORY,8,2010-11-29 15:07:00,14.95,12415.0,Australia
55728,534450,22424,ENAMEL BREAD BIN CREAM,1,2010-11-22 16:10:00,12.75,12393.0,Australia


`nlargest` and `nsmallest` are shortcuts for "sort and take the top n".

In [18]:
df.nlargest(3, "Quantity")[["Invoice", "Description", "Quantity"]]

,Invoice,Description,Quantity
7253,530715,ROTATING SILVER ANGELS T-LIGHT HLDR,9360
7254,530715,BROCADE RING PURSE,6336
7250,530714,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4320


### A new column from old ones

Arithmetic between two columns works row by row, with no loop. The result is a new
Series. Assigning it to a new name adds a column to the table.

In [19]:
df["line_revenue"] = df["Quantity"] * df["Price"]
df[["Quantity", "Price", "line_revenue"]].head()

,Quantity,Price,line_revenue
0,6,7.95,47.7
1,4,7.95,31.8
2,10,6.75,67.5
3,25,0.42,10.5
4,4,7.65,30.6


In [ ]:
# Predict: the shape was (78015, 8) when the file was loaded. What is it now?
df.shape

<details>
<summary>Answer</summary>

Nine columns, the same number of rows. A new column never changes what one row is.

</details>

### Names

- Lowercase words joined by underscores: `line_revenue`.
- `df` for the one main table. When there are two, a name that says what each holds.
- A condition is named as a question: `is_cancelled`, `has_customer`.
- A filtered table is named for what it holds: `germany`. Never `df2` or `temp`.
- Column names stay as they are in the file.

**Try it.** Add a column `is_cancelled` that is `True` when the invoice number starts
with the letter C, then count the cancelled rows. `.str` is the door to text methods:
`df["Invoice"].str.startswith("C")` asks every value in the column whether it starts
with C, and gives a column of True and False.

In [21]:
# Try it: replace the comment below with the assignment, then count with .sum()
# df["is_cancelled"] = ...

## Part 3 — What a column is stored as, and what it holds

`dtypes` shows the storage type of every column. The storage type decides which
operations work. It does not tell you what the values mean.

In [22]:
df.dtypes

Invoice             str
StockCode           str
Description         str
Quantity          int64
InvoiceDate         str
Price           float64
Customer ID     float64
Country             str
line_revenue    float64
dtype: object

`int64` and `float64` are numbers. `object` in older pandas versions and `str` in newer
ones both mean text. Two columns deserve a second look: `InvoiceDate` and `Customer ID`.

### Dates

In [ ]:
# Predict: what does this return?
df["InvoiceDate"].dt.day.head()

<details>
<summary>Answer</summary>

An `AttributeError`. `.dt` is the door to date methods, the way `.str` is for text:
`.dt.day` gives the day of the month of every value, `.dt.day_name()` the weekday. It
works only on datetime columns,
and `InvoiceDate` is text.

</details>

The file stores dates as text, and reading the file does not convert them, because
pandas does not guess. Convert on purpose:

In [24]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDate"].head(3)

0   2010-11-01 08:56:00
1   2010-11-01 08:56:00
2   2010-11-01 08:56:00
Name: InvoiceDate, dtype: datetime64[us]

In [25]:
df["InvoiceDate"].dt.day_name().value_counts()

InvoiceDate
Monday       15872
Tuesday      14713
Thursday     13482
Wednesday    12059
Sunday       11605
Friday       10284
Name: count, dtype: int64

`value_counts()` counts how many times each value appears. Now calendar questions work: which weekday, which hour, the first and the last date. One
day of the week is missing from the counts. The table cannot tell you why. That is a
question for the shop.

In [26]:
print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

2010-11-01 08:56:00
2010-11-30 19:35:00


### A whole number stored as a decimal

`Customer ID` holds values like `16316.0`. An id with a `.0` means the column is stored
as decimals.

In [ ]:
# Predict: what happens?
df["Customer ID"].astype(int)

<details>
<summary>Answer</summary>

An error. The column has missing values, and the plain integer type has no way to store
a missing value. That is why pandas stored the column as `float64` when it read the file:
the float type has `NaN`.

</details>

`NaN` stands for "not a number" and is how pandas marks a missing value. The integer type
`Int64`, with a capital I, allows missing values too, and prints them as `<NA>`. It holds
both:

In [28]:
df["Customer ID"] = df["Customer ID"].astype("Int64")
df["Customer ID"].head()

0    16316
1    16316
2    16316
3    16316
4    16316
Name: Customer ID, dtype: Int64

The rule: **a whole-number column with missing values turns into decimals.** When an id
column prints with `.0`, look for missing values.

### Conversion can create missing values

`Invoice` is text because some values start with C. Force it to numbers and see what
happens to those values:

In [ ]:
# Predict: how many missing values will the converted column have?
pd.to_numeric(df["Invoice"], errors="coerce").isna().sum()

<details>
<summary>Answer</summary>

`errors="coerce"` turns every value that is not a number into `NaN`. The count equals the
number of rows whose invoice number starts with C: `df["Invoice"].str.startswith("C").sum()`
gives the same number. The conversion silently deleted the
information that made those rows special. The count of new missing values tells you how
many values it threw away.

</details>

## Part 4 — The profiling sequence

Run these in this order on every table you have not met before.

### Step 1 — Size and a look

In [30]:
df.shape

(78015, 9)

In [31]:
df.sample(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_revenue
67614,535421,22737,RIBBON REEL CHRISTMAS PRESENT,1,2010-11-26 11:50:00,1.65,13555,United Kingdom,1.65
74273,536010,22178,VICTORIAN GLASS HANGING T-LIGHT,12,2010-11-29 15:18:00,1.25,16592,United Kingdom,15.00
11468,531036,90114,SUMMER DAISIES BAG CHARM,1,2010-11-05 11:58:00,2.51,<NA>,United Kingdom,2.51
20148,531789,22730,ALARM CLOCK BAKELIKE IVORY,2,2010-11-09 15:07:00,3.75,15570,United Kingdom,7.50
30343,532636,21179,NO JUNK MAIL METAL SIGN,12,2010-11-12 15:46:00,1.25,12635,Germany,15.00


`head` shows the top of the file, which is often the oldest or the cleanest part.
`sample` shows rows from anywhere.

### Step 2 — What is one row

Say it in words: one row is one line on one receipt. That sentence is the **grain** of
the table, what one row stands for. Then check it with numbers. If a receipt and a row
were the same thing, the two counts would be equal. `nunique()` counts the distinct
values.

In [32]:
print(len(df))
print(df["Invoice"].nunique())

78015
3669


If one row is one line on one receipt, a receipt-and-product pair never repeats.
`duplicated` marks a row that repeats an earlier one; `subset` names the columns that
must match.

In [33]:
df.duplicated(subset=["Invoice", "StockCode"]).sum()

np.int64(2920)

The count says how many rows repeat a receipt-and-product pair seen earlier in the
table. A count above zero means the stated grain is not exact. Write it down and carry
on.

### Step 3 — Types

`dtypes` after the two conversions:

In [34]:
df.dtypes

Invoice                    str
StockCode                  str
Description                str
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID              Int64
Country                    str
line_revenue           float64
dtype: object

### Step 4 — Missing values

In [35]:
df.isna().sum()

Invoice             0
StockCode           0
Description       312
Quantity            0
InvoiceDate         0
Price               0
Customer ID     16525
Country             0
line_revenue        0
dtype: int64

`isna()` works on one column and on the whole table. On the table, `.sum()` gives one
count per column.

In [36]:
df.isna().mean().round(3)

Invoice         0.000
StockCode       0.000
Description     0.004
Quantity        0.000
InvoiceDate     0.000
Price           0.000
Customer ID     0.212
Country         0.000
line_revenue    0.000
dtype: float64

The counts show which columns have gaps. The shares show how large the gaps are.

In [ ]:
# Predict: will the missing values appear in this output?
df["Customer ID"].value_counts().head(3)

<details>
<summary>Answer</summary>

They do not. `value_counts()` drops missing values by default, so the most common
"value" in this column, the missing one, is invisible.

</details>

Ask for it:

In [38]:
df["Customer ID"].value_counts(dropna=False).head(3)

Customer ID
<NA>     16525
14911      654
12748      555
Name: count, dtype: Int64

`<NA>` is the missing marker of the `Int64` type; it means the same as `NaN`.

The rule: **when you count values, ask whether missing was counted.**

### Step 5 — Duplicates

In [ ]:
# Predict: which count is larger, the first or the second, and why?
print(df.duplicated().sum())
print(df.duplicated(keep=False).sum())

<details>
<summary>Answer</summary>

`duplicated()` marks every row that repeats an earlier row, so the first count is the
number of extra copies. `keep=False` marks every member of a repeated group, the first
copy included, so the second count is larger. Use the first count when you ask "how many
rows would dropping duplicates remove".

</details>

Use the second when you want to look at the groups:

In [40]:
df[df.duplicated(keep=False)].sort_values(["Invoice", "StockCode"]).head(6)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_revenue
245,530014,21033,JUMBO BAG CHARLIE AND LOLA TOYS,1,2010-11-01 10:37:00,2.95,17238,United Kingdom,2.95
249,530014,21033,JUMBO BAG CHARLIE AND LOLA TOYS,1,2010-11-01 10:37:00,2.95,17238,United Kingdom,2.95
252,530014,21645,ASSORTED TUTTI FRUTTI ROUND BOX,1,2010-11-01 10:37:00,1.65,17238,United Kingdom,1.65
268,530014,21645,ASSORTED TUTTI FRUTTI ROUND BOX,1,2010-11-01 10:37:00,1.65,17238,United Kingdom,1.65
274,530014,22968,ROSE COTTAGE KEEPSAKE BOX,1,2010-11-01 10:37:00,9.95,17238,United Kingdom,9.95
280,530014,22968,ROSE COTTAGE KEEPSAKE BOX,1,2010-11-01 10:37:00,9.95,17238,United Kingdom,9.95


Look at the six rows: each pair shares the receipt number and the time, to the minute.
That can be one line recorded twice, or the same product added to the basket twice
within one minute. The table cannot tell you which. Count them, look at them, and do not
drop them yet.

### Step 6 — Distributions

In [41]:
df.describe().round(2)

,Quantity,InvoiceDate,Price,Customer ID,line_revenue
count,78015.00,78015,78015.00,61490.0,78015.00
mean,8.64,2010-11-16 10:47:08.636800,3.87,15468.17,18.24
min,-9000.00,2010-11-01 08:56:00,0.00,12351.0,-6706.71
25%,1.00,2010-11-09 13:50:00,1.25,14085.0,3.36
50%,3.00,2010-11-16 13:35:00,2.10,15547.0,8.40
75%,9.00,2010-11-23 16:08:00,4.21,16923.0,16.98
max,9360.00,2010-11-30 19:35:00,6706.71,18287.0,15818.40
std,77.04,NaN,36.58,1687.03,93.60


Read four things in this table: the minimum, the maximum, the median (the 50% row) and
the count. Here the minimum quantity is negative, the maximum is thousands of times the
median, and the minimum price is zero. Each of these is a question to write down.

`describe()` skips the text columns unless you ask for them. `include="all"` adds them,
with a different set of rows: the count, the number of distinct values, the most common
value and how often it appears. A row that does not apply to a column shows `NaN`, or
`<NA>` for the `Int64` column.

In [42]:
df.describe(include="all")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_revenue
count,78015,78015,77703,78015.000000,78015,78015.000000,61490.0,78015,78015.000000
unique,3669,3137,3089,NaN,NaN,NaN,<NA>,29,NaN
top,536031,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,<NA>,United Kingdom,NaN
freq,582,423,430,NaN,NaN,NaN,<NA>,72438,NaN
mean,NaN,NaN,NaN,8.637518,2010-11-16 10:47:08.636800,3.869946,15468.172955,NaN,18.235655
min,NaN,NaN,NaN,-9000.000000,2010-11-01 08:56:00,0.000000,12351.0,NaN,-6706.710000
25%,NaN,NaN,NaN,1.000000,2010-11-09 13:50:00,1.250000,14085.0,NaN,3.360000
50%,NaN,NaN,NaN,3.000000,2010-11-16 13:35:00,2.100000,15547.0,NaN,8.400000
75%,NaN,NaN,NaN,9.000000,2010-11-23 16:08:00,4.210000,16923.0,NaN,16.980000
max,NaN,NaN,NaN,9360.000000,2010-11-30 19:35:00,6706.710000,18287.0,NaN,15818.400000


Two things to read here: `Country` has 29 distinct values and one of them covers most
rows, and `Description` has 3 089 distinct values, which is close to one per product.

In [43]:
df["Country"].value_counts(normalize=True).head(5).round(3)

Country
United Kingdom    0.929
EIRE              0.014
Germany           0.013
France            0.013
Netherlands       0.006
Name: proportion, dtype: float64

`normalize=True` turns counts into shares. For a text column, the share of each value
shows how the rows are spread. Here one country
holds 93% of the rows. Any average over the whole table is mostly an average over that
country.

### Step 7 — Ranges and impossible values

Write the condition for each value that should not exist, and count.

In [44]:
print("negative quantity:", (df["Quantity"] < 0).sum())
print("zero price:       ", (df["Price"] == 0).sum())
print("cancelled invoice:", df["Invoice"].str.startswith("C").sum())

negative quantity: 1419
zero price:        357
cancelled invoice: 1194


The negative quantities outnumber the rows with a cancelled invoice number. So some
negative rows are not cancellations. Look at what they say:

In [45]:
not_cancelled = (df["Quantity"] < 0) & ~df["Invoice"].str.startswith("C")
df.loc[not_cancelled, "Description"].value_counts(dropna=False).head(6)

Description
NaN                  199
?                      7
damaged                5
damages                4
Damages                2
code mix up 72597      1
Name: count, dtype: int64

Most of them have no description at all. The rest say damaged, damages, a question
mark, or a note. These rows record something other than a sale, a correction to the
shop's records of what it has in storage, and the table says no more than that. A value that looks impossible usually marks a
different kind of record.

### Step 8 — Consistency

Compare the number of distinct values before and after cleaning: `.str.strip()` removes
spaces at both ends, `.str.upper()` makes every letter a capital, and the two run one
after the other.

In [ ]:
# Predict: will the second number be smaller, equal, or larger than the first?
print(df["Description"].nunique())
print(df["Description"].str.strip().str.upper().nunique())

<details>
<summary>Answer</summary>

Smaller. The same thing written two ways counts as two things: eight names collapse into
names that already exist, with a space at the end or small letters instead of capitals.

</details>

Find one. `unique()` lists each different value once. Every spelling that cleans to
the same name:

In [47]:
cleaned = df["Description"].str.strip().str.upper()
df.loc[cleaned == "BATHROOM METAL SIGN", "Description"].unique()

<StringArray>
['BATHROOM METAL SIGN', 'BATHROOM METAL SIGN ']
Length: 2, dtype: str

### The profile

The eight steps end in a short note. For this table it reads:

- One row is one line on one receipt. The receipt-and-product pair repeats 2 920 times, so
  that grain is not exact.
- 78 015 rows, 8 columns, one month.
- Types converted: the date from text to datetime, the customer id to `Int64`.
- Missing: the customer id in 21% of rows, the description in 0.4%.
- Duplicates: 1 431 exact copies of an earlier row.
- Values that need a decision before any total: 1 419 negative quantities, of which
  1 194 are cancellations; 357 zero prices.
- Questions the table cannot answer: what the negative rows without a cancellation are;
  why there are no Saturdays.

## Part 5 — Wrong answers that raise no error

Three you have met: a label read as a position (Part 1), a whole number turned into a
decimal (Part 3), missing values dropped from a count (Part 4). Two more.

### Changing values through a filter

Work on a copy: `copy()` makes a separate table, so the rest of the notebook is not
affected.

In [ ]:
demo = df.copy()

# Predict: after this cell, how many rows in demo have Price == 0?
demo[demo["Price"] == 0]["Price"] = 0.01
(demo["Price"] == 0).sum()

<details>
<summary>Answer</summary>

The same count as before, and a warning in red. `demo[demo["Price"] == 0]` returns a new
table, the assignment changed that new table, and the new table was thrown away. The
warning says so. Its name depends on the pandas version, `ChainedAssignmentError` or
`SettingWithCopyWarning`, and the cell still ran.

</details>

To change values inside a table, name the rows and the column in one `.loc`:

In [49]:
demo.loc[demo["Price"] == 0, "Price"] = 0.01
(demo["Price"] == 0).sum()

np.int64(0)

### Arithmetic aligns by label

In [50]:
a = df.loc[df["Country"] == "France", "Quantity"].head(3)
b = df.loc[df["Country"] == "Germany", "Quantity"].head(3)
print(a)
print(b)

977     -1
978     -1
4432    36
Name: Quantity, dtype: int64
1599    4
1600    3
1601    6
Name: Quantity, dtype: int64


In [ ]:
# Predict: three sums, or something else?
a + b

<details>
<summary>Answer</summary>

Six rows of `NaN`. pandas adds by label, not by position. The value at label 977 in `a`
is added to the value at label 977 in `b`. There is none, so the result is missing.

</details>

This matching by label is called **alignment**. It is what makes
`df["Quantity"] * df["Price"]` in Part 2 safe: both columns share every label. To add by position, reset both indexes first.

### The five traps in one table

| Trap | What you see | What to do |
|---|---|---|
| Label read as position | `.loc[0]` fails, or returns another row, after sorting | `.iloc` for positions, `reset_index(drop=True)` after sorting |
| Whole numbers turned into decimals | an id column prints as `16316.0` | look for missing values, convert to `Int64` |
| Missing dropped from a count | `value_counts()` shows no `NaN` | `value_counts(dropna=False)` |
| Changing values through a filter | a warning, and nothing changes | `df.loc[condition, "column"] = value` |
| Arithmetic aligns by label | `NaN` where you expected numbers | share one index, or reset both |

## Part 6 — What the checks answer, and what they cannot

The eight steps answer questions about the **integrity** of the table: whether it is
in good order inside.

| The question | Where it is answered |
|---|---|
| Is it complete? | Step 1 size, Step 4 missing, Step 5 duplicates |
| Is it consistent? | Step 8 |
| Is it at the grain the question needs? | Step 2 |
| Are the values what the columns claim? | Step 3 types, Step 6 distributions, Step 7 ranges |
| Is it current? | the first and last date, Part 3 |

They do not answer whether the table **fits** the question you want to ask. Is one month
typical? Does this shop resemble the shop your question is about? Does a missing customer id
mean an anonymous sale, or a lost record? Those answers come from knowing where the data
came from. The table can hint at them, by comparing rows with and without an id, but it
cannot settle them.

The habit: before any computation, run the sequence and note what you fixed in the table
and what you could not decide from it. The second kind goes to the people who collected
the data.

**Bring to the lecture:** one question this table could not answer for you.

## Summary

- Pick a column with `df["name"]`, rows with a condition inside `df[...]`, and both at
  once with `df.loc[condition, columns]`.
- A condition produces a boolean mask, a Series of True and False. `.sum()` on the mask
  counts the matches, `.mean()` gives their share.
- `.loc` uses labels, `.iloc` uses positions. After sorting they differ.
- The storage type decides what works. A date stored as text and an id stored as a decimal
  are the two signs to look for.
- Eight steps before any computation: size, grain, types, missing, duplicates,
  distributions, ranges, consistency. Then write the profile.